<img src="https://res.cloudinary.com/dtizipxds/image/upload/q_auto/f_auto/v1776264397/banner_yvwajv.png" width="100%">


In [ ]:
%pip install numpy pandas matplotlib seaborn scikit-learn scipy


# Solution - Clustering


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from scipy.cluster.hierarchy import linkage, dendrogram

sns.set_theme(style="whitegrid")
train_df = pd.read_csv(Path("../../Exercises/Clustering/train.csv"))
test_df = pd.read_csv(Path("../../Exercises/Clustering/test.csv"))
sol_df = pd.read_csv(Path("solution.csv"))

feature_cols = ["age", "annual_income_k", "spending_score"]
X_train = train_df[feature_cols]
X_test = test_df[feature_cols]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
kmeans = KMeans(n_clusters=5, random_state=42, n_init=20)
kmeans.fit(X_train_scaled)
test_kmeans = kmeans.predict(X_test_scaled)

dbscan = DBSCAN(eps=0.9, min_samples=6)
dbscan_labels = dbscan.fit_predict(X_test_scaled)

agg = AgglomerativeClustering(n_clusters=5, linkage="ward")
agg_labels = agg.fit_predict(X_test_scaled)

out = pd.DataFrame({
    "cluster_kmeans_5": test_kmeans,
    "cluster_dbscan": dbscan_labels,
    "cluster_agg": agg_labels,
})
out.head()


In [ ]:
# Quick check against reference labels from solution.csv
if "cluster_kmeans_5" in sol_df.columns:
    same_ratio = (sol_df["cluster_kmeans_5"].values == test_kmeans).mean()
    print("KMeans match with reference:", round(float(same_ratio), 4))


In [ ]:
plot_df = test_df.copy()
plot_df["kmeans_cluster"] = test_kmeans

plt.figure(figsize=(8, 5))
sns.scatterplot(data=plot_df, x="annual_income_k", y="spending_score", hue="kmeans_cluster", s=70)
plt.title("KMeans Clusters on Test Set")
plt.tight_layout()
plt.show()

linked = linkage(X_train_scaled, method="ward")
plt.figure(figsize=(10, 5))
dendrogram(linked, truncate_mode="lastp", p=20, show_leaf_counts=True)
plt.title("Train Dendrogram")
plt.tight_layout()
plt.show()
